# Ray Unit 4 Capstone - Colab Remote Docker Launcher

Use this notebook from Google Colab when the live demo needs Docker execution. Colab is used as the notebook/control environment. The Docker daemon runs on a separate Linux host or VM that has Docker Engine and the Docker Compose plugin installed.

Colab itself is not the Docker host.

## What This Notebook Does

1. Shows the Colab Linux runtime information.
2. Configures SSH access to a remote Linux Docker host.
3. Verifies Docker Engine and Docker Compose on the remote host.
4. Uploads the required Ray course files to the remote host.
5. Runs the Docker-based Ray cluster flow on the remote host.
6. Copies the cluster artifacts back to Colab for inspection.

## 1. Show Colab Runtime

This cell proves that the notebook is running on a Linux runtime. It does not prove Docker is available, because Docker must run on the remote Linux host.

In [ ]:
!uname -a
!cat /etc/os-release | sed -n '1,6p'
!python --version

## 2. Configure Remote Docker Host

Fill these values before running the rest of the notebook.

The SSH user must be able to run `docker` without an interactive password prompt.

`REMOTE_WORKDIR` is where the project copy will be created on the remote Linux host.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess

REMOTE_USER = ""          # example: "ubuntu"
REMOTE_HOST = ""          # example: "203.0.113.10"
REMOTE_PORT = 22
REMOTE_WORKDIR = "/tmp/open_u_tools"
WORKERS = 2

unsafe_workdirs = {"", "/", "/tmp", "/home", "/workspace"}
if REMOTE_WORKDIR.rstrip("/") in unsafe_workdirs:
    raise ValueError("REMOTE_WORKDIR must be a dedicated project directory, for example /tmp/open_u_tools.")

# Optional: set a private key string here, or upload/write ~/.ssh/capstone_key manually.
SSH_PRIVATE_KEY = ""

## 3. Prepare SSH Helper

This cell writes the optional SSH key and defines helper functions for remote commands and file transfer.

In [ ]:
ssh_dir = Path.home() / ".ssh"
ssh_dir.mkdir(mode=0o700, exist_ok=True)
key_path = ssh_dir / "capstone_key"

if SSH_PRIVATE_KEY.strip():
    key_path.write_text(SSH_PRIVATE_KEY.strip() + "\n", encoding="utf-8")
    key_path.chmod(0o600)

if not REMOTE_USER or not REMOTE_HOST:
    raise ValueError("Set REMOTE_USER and REMOTE_HOST before continuing.")

remote = f"{REMOTE_USER}@{REMOTE_HOST}"
ssh_base = ["ssh", "-o", "StrictHostKeyChecking=no", "-p", str(REMOTE_PORT)]
scp_base = ["scp", "-o", "StrictHostKeyChecking=no", "-P", str(REMOTE_PORT)]
if key_path.exists():
    ssh_base.extend(["-i", str(key_path)])
    scp_base.extend(["-i", str(key_path)])

def run_local(args: list[str]) -> None:
    print("+", " ".join(shlex.quote(arg) for arg in args), flush=True)
    subprocess.run(args, check=True)

def run_remote(command: str) -> None:
    run_local(ssh_base + [remote, command])

## 4. Verify Remote Docker Engine

This is the Docker evidence for the live demo. These commands must run on the remote host.

In [ ]:
run_remote("docker version")
run_remote("docker compose version")
run_remote("docker info --format 'Engine={{.ServerVersion}}; OS={{.OperatingSystem}}; OSType={{.OSType}}; CPUs={{.NCPU}}'")

## 5. Locate Project Files In Colab

This launcher expects the `Ray` folder to exist under `PROJECT_ROOT`. Upload the project folder to Colab or clone it before this cell.

In [ ]:
PROJECT_ROOT = Path("/content/OpenU/Tools")

required_paths = [
    PROJECT_ROOT / "Ray" / "environment.yml",
    PROJECT_ROOT / "Ray" / "1_cluster_setup" / "Dockerfile",
    PROJECT_ROOT / "Ray" / "1_cluster_setup" / "docker-compose.yml",
    PROJECT_ROOT / "Ray" / "4_ray_capstone_project" / "solution" / "01_download_real_data.ipynb",
    PROJECT_ROOT / "Ray" / "4_ray_capstone_project" / "solution" / "02_prepare_assets.ipynb",
    PROJECT_ROOT / "Ray" / "4_ray_capstone_project" / "solution" / "03_run_replay.ipynb",
    PROJECT_ROOT / "Ray" / "4_ray_capstone_project" / "solution" / "run_on_docker_engine.sh",
]

missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required project files:\n" + "\n".join(missing))

print("Project files found under", PROJECT_ROOT)

## 6. Upload Project Files To Remote Host

Only the files needed for the Docker cluster run are packed and copied.

In [ ]:
archive_path = Path("/content/ray_capstone_remote_upload.tgz")
tar_args = [
    "tar", "-C", str(PROJECT_ROOT), "-czf", str(archive_path),
    "Ray/environment.yml",
    "Ray/1_cluster_setup/Dockerfile",
    "Ray/1_cluster_setup/docker-compose.yml",
    "Ray/4_ray_capstone_project/solution/README.md",
    "Ray/4_ray_capstone_project/solution/01_download_real_data.ipynb",
    "Ray/4_ray_capstone_project/solution/02_prepare_assets.ipynb",
    "Ray/4_ray_capstone_project/solution/03_run_replay.ipynb",
    "Ray/4_ray_capstone_project/solution/run_on_docker_engine.sh",
]
run_local(tar_args)

run_remote(f"rm -rf {shlex.quote(REMOTE_WORKDIR)} && mkdir -p {shlex.quote(REMOTE_WORKDIR)}")
run_local(scp_base + [str(archive_path), f"{remote}:{REMOTE_WORKDIR}/ray_capstone_remote_upload.tgz"])
run_remote(f"tar -xzf {shlex.quote(REMOTE_WORKDIR)}/ray_capstone_remote_upload.tgz -C {shlex.quote(REMOTE_WORKDIR)}")

## 7. Run Docker-Based Ray Cluster Flow

This starts the virtual Docker-based Ray cluster on the remote host and executes the three solution notebooks through Ray Jobs.

In [ ]:
remote_solution = f"{REMOTE_WORKDIR}/Ray/4_ray_capstone_project/solution"
run_remote(f"cd {shlex.quote(remote_solution)} && bash ./run_on_docker_engine.sh --workers {WORKERS}")

## 8. Ray Dashboard

The Docker cluster exposes the Ray dashboard on the remote host port `8265`. If the VM firewall allows it, open `http://<REMOTE_HOST>:8265`. If the port is not public, create an SSH tunnel from your own machine to the remote host and open `http://127.0.0.1:8265`.

## 9. Copy Artifacts Back To Colab

The remote Docker run writes artifacts under `Ray/1_cluster_setup/head_workspace/ray_capstone`. This cell copies them back to `/content/ray_capstone_remote_outputs`.

In [ ]:
local_artifact_dir = Path("/content/ray_capstone_remote_outputs")
local_artifact_dir.mkdir(parents=True, exist_ok=True)
remote_artifact_dir = f"{REMOTE_WORKDIR}/Ray/1_cluster_setup/head_workspace/ray_capstone"
run_local(scp_base + ["-r", f"{remote}:{remote_artifact_dir}/.", str(local_artifact_dir)])
print("Artifacts copied to", local_artifact_dir)
!find /content/ray_capstone_remote_outputs -maxdepth 3 -type f | sort | head -80